<a href="https://colab.research.google.com/github/asnapareeth004-hue/encoder_decoder/blob/main/lstm_text_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. IMPORT LIBRARIES

import torch
import torch.nn as nn
import torch.optim as optim
import re
from collections import Counter

In [ ]:
# 2. SAMPLE TEXT DATASET

text = """
the sun rises in the morning
the sun shines brightly
the moon shines at night
the stars shine in the sky
machine learning is a part of artificial intelligence
deep learning uses neural networks
lstm is a type of recurrent neural network
neural networks learn patterns from data
python is a popular programming language
deep learning can generate text
the computer processes the input data
artificial intelligence is changing the world
"""

# Convert text into words
words = re.findall(r"\b\w+\b", text.lower())

# Create vocabulary
vocab = sorted(set(words))

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

vocab_size = len(vocab)

# Convert words into numerical values
tokens = [word_to_idx[word] for word in words]

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 45


In [ ]:
# 3. PREPARE TRAINING DATA

sequence_length = 4

X = []
Y = []

for i in range(len(tokens) - sequence_length):

    X.append(tokens[i:i + sequence_length])
    Y.append(tokens[i + sequence_length])

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

In [ ]:
# 4. DEFINE LSTM MODEL

class LSTMTextGenerator(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 64)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            batch_first=True
        )

        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        output = self.fc(output[:, -1, :])

        return output


model = LSTMTextGenerator(vocab_size)

In [ ]:
# 5. TRAINING SETUP

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

epochs = 200

In [ ]:
# 6. TRAINING

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, Y)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )


Epoch [20/200], Loss: 0.0065
Epoch [40/200], Loss: 0.0004
Epoch [60/200], Loss: 0.0002
Epoch [80/200], Loss: 0.0002
Epoch [100/200], Loss: 0.0002
Epoch [120/200], Loss: 0.0001
Epoch [140/200], Loss: 0.0001
Epoch [160/200], Loss: 0.0001
Epoch [180/200], Loss: 0.0001
Epoch [200/200], Loss: 0.0001


In [ ]:
# 7. TEXT GENERATION

def generate_text(start_text, next_words=10):

    model.eval()

    result = start_text.lower().split()

    with torch.no_grad():

        for _ in range(next_words):

            # Convert input words to numerical values
            input_words = result[-sequence_length:]

            input_tokens = [
                word_to_idx.get(word, 0)
                for word in input_words
            ]

            # Pad if fewer than sequence_length words
            if len(input_tokens) < sequence_length:
                input_tokens = (
                    [0] * (sequence_length - len(input_tokens))
                    + input_tokens
                )

            input_tensor = torch.tensor(
                [input_tokens],
                dtype=torch.long
            )

            # Predict next word
            output = model(input_tensor)

            predicted = torch.argmax(output, dim=1).item()

            next_word = idx_to_word[predicted]

            result.append(next_word)

    return " ".join(result)



In [ ]:
# 8. GENERATE SENTENCES

print("\nGenerated Text:")

print(generate_text("the sun", 8))

print(generate_text("deep learning", 8))

print(generate_text("neural networks", 8))


Generated Text:
the sun shines brightly the moon shines at night the
deep learning neural networks learn patterns from data python is
neural networks lstm is a type of recurrent neural network
